In [ ]:
# Fooocus on Kaggle. Before running: right panel -> Session options -> Accelerator: GPU T4 x2 (or P100), Internet: ON.
import os, shutil, subprocess
if shutil.which('nvidia-smi') is None or subprocess.run(['nvidia-smi'], capture_output=True).returncode != 0:
    raise SystemExit('No GPU found. Session options -> Accelerator -> GPU T4 x2, then run again.')

# Fooocus needs Python 3.10 with pinned packages, so it runs in its own environment created with uv.
!pip install -q uv
!uv venv -q --seed --python 3.10 /tmp/venv
!uv pip install -q --python /tmp/venv/bin/python pygit2==1.15.1
!which ffmpeg > /dev/null || (apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null)
%cd /kaggle/working
!git clone https://github.com/Koushikdama/FOOOCUS.git Fooocus
%cd /kaggle/working/Fooocus
!uv pip install -q --python /tmp/venv/bin/python "diffusers>=0.31.0" "imageio[ffmpeg]>=2.34.0" "imageio-ffmpeg>=0.5.1"

# Launch. Kaggle has no port proxy, so open the https://....gradio.live link printed below
# (after "App started successful"). Results are saved under /kaggle/working/Fooocus/outputs (Output panel).
env = {k: v for k, v in os.environ.items() if k not in ('PYTHONPATH', 'PYTHONHOME')}
env.update(MPLBACKEND='Agg', PYTHONUNBUFFERED='1', TQDM_MININTERVAL='3')
proc = subprocess.Popen(['/tmp/venv/bin/python', 'entry_with_update.py', '--share', '--always-high-vram'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                        cwd='/kaggle/working/Fooocus', env=env)
try:
    for line in proc.stdout:
        print(line, end='')
        if 'Running on public URL' in line:
            print('\n==> Open this link once the log says "App started successful":', line.split()[-1], '\n')
finally:
    proc.terminate()